<a href="https://colab.research.google.com/github/aansikkaw/Bio-BERT-Medical-Entity-Resolution-Agentic-Healthcare-System/blob/main/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install transformers torch datasets scikit-learn pandas numpy

import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load Bio-BERT model
model_name = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"Model loaded: {model_name}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# Prepare sample training data
# Replace with your actual medical entity data
train_data = {
    'text': [
        'Patient with hypertension and diabetes',
        'Administered ibuprofen 500mg',
        'Normal chest X-ray',
    ],
    'label': [1, 0, 1]
}

train_dataset = Dataset.from_dict(train_data)

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, padding='max_length')

train_dataset = train_dataset.map(preprocess_function, batched=True)
train_dataset = train_dataset.remove_columns(['text'])
train_dataset = train_dataset.rename_column('label', 'labels')

print(f"Dataset prepared. Size: {len(train_dataset)}")

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=100,
    save_total_limit=2,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

# Fine-tune the model
trainer.train()

print("Training completed!")